# Session 27 — SAM 3 in Practice 🎯
### Promptable Concept Segmentation with Meta's Segment Anything Model 3

In the theory part of this session we saw that **SAM 3** answers a new question:

> SAM 1 / SAM 2: *"segment **this thing** I'm pointing at"* → one object per prompt
>
> **SAM 3**: *"segment **every instance** of this concept"* → all matching objects, from a **text phrase** or an **exemplar box**

In this notebook we will:
1. Set up SAM 3 through Hugging Face Transformers
2. Segment objects with a **text prompt** (a short noun phrase)
3. Understand the outputs: **masks, boxes, and confidence scores**
4. Prompt with a **visual exemplar** (a bounding box)
5. **Combine** text with negative boxes to refine a concept

---
### ⚠️ Before you run: two prerequisites

1. **Model access** — `facebook/sam3` is a *gated* model.
   Visit **https://huggingface.co/facebook/sam3**, click *"Request access"*, and wait for approval (usually quick).
2. **GPU runtime** — in Colab: `Runtime → Change runtime type → T4 GPU`.
   The model has 848M parameters; a T4 is enough for image inference.


## 1. Setup

We install 🤗 Transformers (SAM 3 support was added in Nov 2025) and log in to Hugging Face so we can download the gated checkpoint.

In [ ]:
# Install dependencies (quiet mode)
!pip install -q --upgrade transformers accelerate pillow matplotlib

In [ ]:
import torch

# Check that we have a GPU
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
if device == "cpu":
    print("⚠️ No GPU detected — inference will be slow. In Colab: Runtime → Change runtime type → T4 GPU")

In [ ]:
# Log in to Hugging Face (needed for the gated facebook/sam3 checkpoint).
# Paste an access token from https://huggingface.co/settings/tokens
from huggingface_hub import notebook_login

notebook_login()

## 2. Load SAM 3

One model, one processor. The **processor** handles image resizing, text tokenization, and turning raw model outputs back into pixel-space masks. The **model** is the 848M-parameter detector + tracker we discussed in the slides.

*The first run downloads ~3.5 GB of weights — this takes a couple of minutes.*

In [ ]:
from transformers import Sam3Model, Sam3Processor

model = Sam3Model.from_pretrained("facebook/sam3").to(device)
processor = Sam3Processor.from_pretrained("facebook/sam3")

n_params = sum(p.numel() for p in model.parameters())
print(f"Model loaded ✓  ({n_params/1e6:.0f}M parameters)")

## 3. Helper: visualizing results

SAM 3 returns, for each detected instance:
- a **binary mask** (which pixels belong to the object),
- a **bounding box** in `[x1, y1, x2, y2]` pixel coordinates,
- a **confidence score** in `[0, 1]`.

The helper below overlays every mask in a different color, and draws the box + score. We will reuse it throughout the notebook.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches


def show_results(image, results, title=""):
    """Overlay instance masks, boxes and scores on the image."""
    img = np.array(image)
    fig, ax = plt.subplots(figsize=(10, 7))
    ax.imshow(img)

    masks = results["masks"]          # (N, H, W) binary masks
    boxes = results["boxes"]          # (N, 4) xyxy pixel coords
    scores = results["scores"]        # (N,) confidence

    rng = np.random.default_rng(42)   # fixed seed → same colors every run
    for mask, box, score in zip(masks, boxes, scores):
        color = rng.random(3) * 0.7 + 0.3          # avoid too-dark colors
        m = mask.cpu().numpy().astype(bool)

        # colored, semi-transparent mask overlay
        overlay = np.zeros((*m.shape, 4))
        overlay[m] = [*color, 0.55]
        ax.imshow(overlay)

        # bounding box + score label
        x1, y1, x2, y2 = box.tolist()
        ax.add_patch(mpatches.Rectangle((x1, y1), x2 - x1, y2 - y1,
                                        fill=False, edgecolor=color, linewidth=2))
        ax.text(x1, y1 - 5, f"{score:.2f}", color="white", fontsize=10,
                bbox=dict(facecolor=tuple(color), alpha=0.9, pad=1))

    ax.set_title(f"{title}  —  {len(masks)} instance(s) found")
    ax.axis("off")
    plt.tight_layout()
    plt.show()

print("Helper ready ✓")

## 4. Load a test image

We use images from the COCO validation set (the same ones used in the official SAM 3 documentation), fetched directly from the web.

In [ ]:
import requests
from PIL import Image

cat_url = "http://images.cocodataset.org/val2017/000000077595.jpg"
kitchen_url = "http://images.cocodataset.org/val2017/000000136466.jpg"

cat_image = Image.open(requests.get(cat_url, stream=True).raw).convert("RGB")
kitchen_image = Image.open(requests.get(kitchen_url, stream=True).raw).convert("RGB")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].imshow(cat_image); axes[0].set_title("Image A — cat"); axes[0].axis("off")
axes[1].imshow(kitchen_image); axes[1].set_title("Image B — kitchen"); axes[1].axis("off")
plt.show()

## 5. Text prompts — "segment every X"

This is SAM 3's headline feature: give it a **short noun phrase** and it returns a mask for **every** matching instance — no clicks needed.

The pipeline is always the same three steps:
1. `processor(...)` — prepare image + prompt
2. `model(**inputs)` — one forward pass
3. `post_process_instance_segmentation(...)` — back to pixel-space masks

In [ ]:
def segment_with_text(image, prompt, threshold=0.5):
    """Run SAM 3 with a text prompt and return masks/boxes/scores."""
    inputs = processor(images=image, text=prompt, return_tensors="pt").to(device)

    with torch.no_grad():
        outputs = model(**inputs)

    results = processor.post_process_instance_segmentation(
        outputs,
        threshold=threshold,          # confidence cut-off for detections
        mask_threshold=0.5,           # pixel probability → binary mask
        target_sizes=inputs.get("original_sizes").tolist(),
    )[0]
    return results


# Segment every "cat" in image A
results = segment_with_text(cat_image, "cat")
show_results(cat_image, results, 'text prompt: "cat"')

In [ ]:
# The concept can be more specific — a phrase, not just a class name.
# Note how "ear" finds parts, not whole objects.
results = segment_with_text(cat_image, "ear")
show_results(cat_image, results, 'text prompt: "ear"')

### 🔍 Your turn — explore the open vocabulary

SAM 3 was trained on **4 million unique concepts**, so it is not limited to the 80 COCO classes. Try phrases of your own on the kitchen image: `"pot"`, `"knob"`, `"wooden cabinet"`, `"stove burner"`…

**Also try the `threshold` parameter** — lower it to 0.3 and SAM 3 keeps less-confident detections; raise it to 0.7 and only sure hits remain. This is the standard precision/recall trade-off.

In [ ]:
# Experiment here — change the phrase and the threshold!
results = segment_with_text(kitchen_image, "knob", threshold=0.5)
show_results(kitchen_image, results, 'text prompt: "knob"')

## 6. Exemplar prompts — "segment every object LIKE this one"

Sometimes a concept is hard to name. Instead of text, we can show SAM 3 **one example** by drawing a positive box around it — the model then finds **all other instances** of the same concept.

- `input_boxes` — the exemplar box(es), `[x1, y1, x2, y2]` in pixels
- `input_boxes_labels` — `1` = positive exemplar ("like this"), `0` = negative ("not this")

In [ ]:
def segment_with_boxes(image, boxes, labels, text=None, threshold=0.5):
    """Run SAM 3 with exemplar boxes (optionally combined with text)."""
    inputs = processor(
        images=image,
        text=text,
        input_boxes=[boxes],           # [batch, num_boxes, 4]
        input_boxes_labels=[labels],   # [batch, num_boxes]
        return_tensors="pt",
    ).to(device)

    with torch.no_grad():
        outputs = model(**inputs)

    results = processor.post_process_instance_segmentation(
        outputs,
        threshold=threshold,
        mask_threshold=0.5,
        target_sizes=inputs.get("original_sizes").tolist(),
    )[0]
    return results


# Exemplar: box ONE dial on the oven — SAM 3 should find the similar ones.
dial_box = [59, 144, 76, 163]                      # [x1, y1, x2, y2]
results = segment_with_boxes(kitchen_image, [dial_box], [1])
show_results(kitchen_image, results, "exemplar prompt: one dial box → all dials")

## 7. Combined prompts — text + negative box

Prompts **compose**. A classic refinement: *"segment every handle — but not the oven handle"*.

We pass the text `"handle"` plus a **negative** box (`label = 0`) covering the oven handle. SAM 3 excludes that region from the concept.

In [ ]:
# First: all handles, text only
results_all = segment_with_text(kitchen_image, "handle")
show_results(kitchen_image, results_all, 'text prompt: "handle" (everything)')

In [ ]:
# Now: handles EXCEPT the oven handle (negative box)
oven_handle_box = [40, 183, 318, 204]
results_refined = segment_with_boxes(
    kitchen_image,
    boxes=[oven_handle_box],
    labels=[0],                 # 0 = negative → exclude this region's concept
    text="handle",
)
show_results(kitchen_image, results_refined, '"handle" + negative box on the oven handle')

## 8. Bonus — the official Meta repository

Everything above used 🤗 Transformers. Meta's own repo **https://github.com/facebookresearch/sam3** offers the same models with extra capabilities — most importantly **video**: text-prompt an MP4 and SAM 3 detects *and tracks* every matching object across frames (the tracker + memory-bank half of the architecture we saw in the slides).

```python
# pip install git+https://github.com/facebookresearch/sam3.git   (needs Python 3.12+, CUDA 12.6+)
from sam3.model_builder import build_sam3_video_predictor

predictor = build_sam3_video_predictor()
response = predictor.handle_request(request=dict(
    type="start_session", resource_path="my_video.mp4",
))
response = predictor.handle_request(request=dict(
    type="add_prompt", session_id=response["session_id"],
    frame_index=0, text="player in white",
))
```

The repo's `examples/` folder has ready-made notebooks for video, batched inference, and the **SAM 3 Agent** (SAM 3 as a tool for a multimodal LLM — for prompts that need reasoning, like *"the dog furthest from the camera"*). In March 2026, Meta also released improved **SAM 3.1** checkpoints (`facebook/sam3.1`) with faster multi-object tracking.

## 9. Exercises 🏋️

1. **Open vocabulary limits** — find a phrase where SAM 3 *fails* (returns nothing or the wrong object). Is the concept too abstract? Too relational?
2. **Threshold sweep** — run `"handle"` on the kitchen image with `threshold` = 0.3 / 0.5 / 0.7 and count instances. Where is the sweet spot?
3. **Exemplar vs text** — segment the oven dials once via text (`"dial"`) and once via an exemplar box. Which finds more instances? Which is more precise?
4. **Your own image** — upload a photo (`from google.colab import files; files.upload()`) and segment a concept in it.

---
## Recap

| What we did | The API |
|---|---|
| Text prompt → all instances | `processor(images=…, text="…")` |
| Exemplar box prompt | `input_boxes=[[box]], input_boxes_labels=[[1]]` |
| Refine with negatives | `text="…"` + `input_boxes_labels=[[0]]` |
| Pixel-space results | `processor.post_process_instance_segmentation(…)` |

**Key takeaway:** SAM 1/2 segmented *"this thing"* — SAM 3 segments *"every such thing"*, promptable by language, by example, or both.